# UK National-Level Wind Generation Forecast Analysis

## 1. Objectives
- Analyze the error characteristics of the UK wind power generation model.
- Understand variation in error across forecast horizons (0–48h).
- Evaluate reliability of wind generation to meet electricity demand (Capacity Credit analysis).

## 2. Methodology & Assumptions
- **Data Source**: Elexon BMRS (FUELHH for actuals, WINDFOR for forecasts).
- **Date Window**: January 2025 onwards.
- **Resolution**: 30-minute intervals.
- **Merged State**: Datasets merged on `startTime`, filtering for a 0–48h horizon window.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime

sns.set(style="whitegrid")

# Pull data from the live backend API for consistency
BASE_URL = "http://localhost:5000/api/wind"
params = {
    "startDate": "2025-01-01T00:00:00Z",
    "endDate": "2025-03-15T00:00:00Z",
    "horizon": 0
}

try:
    response = requests.get(BASE_URL, params=params)
    data = response.json()
    df = pd.DataFrame(data['timeseries'])
    df['time'] = pd.to_datetime(df['time'])
    df['error'] = df['forecast'] - df['actual']
    df['abs_error'] = df['error'].abs()
    print(f"Retrieved {len(df)} synced data points.")
except Exception as e:
    print(f"Ensure backend is running: {e}")

## 3. Statistical Metrics Analysis
### MAE, RMSE, and P99 Error

In [ ]:
mae = df['abs_error'].mean()
rmse = np.sqrt((df['error']**2).mean())
bias = df['error'].mean()
p99_error = df['abs_error'].quantile(0.99)

print(f"Mean Absolute Error (MAE): {mae:.2f} MW")
print(f"Root Mean Square Error (RMSE): {rmse:.2f} MW")
print(f"Model Bias: {bias:.2f} MW")
print(f"P99 Absolute Error: {p99_error:.2f} MW")

## 4. Operational Reliablity Analysis
### Question: How many MW of wind power can we reliably expect?

To define "reliability," we look at the lower quantiles of historical generation. Unlike typical power plants with high dispatchability, wind is stochastic. However, it still maintains an operational baseline.

**Our Recommendation Logic**:
Based on First Principles, if we want to ensure high reliability (e.g., 95% of the time, generation is above a certain value), we compute the **P95 Availability (Exceedance level)**.

**Subjective Conclusion**: 
We recommend a reliability baseline of **P95 Value** MW. This ensures that for only 5% of the year (or period), the grid would have to seek alternative dispatchable sources (like peaking gas or interconnectors) for this specific capacity slice.

In [ ]:
reliability_levels = [0.99, 0.95, 0.90, 0.50]
quantiles = df['actual'].quantile([1 - lvl for lvl in reliability_levels])

plt.figure(figsize=(10, 6))
sns.kdeplot(df['actual'], fill=True, color='blue', label='Actual Generation Distribution')
for lvl, val in zip(reliability_levels, quantiles):
    plt.axvline(val, color='r', linestyle='--', alpha=0.5)
    plt.text(val, 0, f"P{int(lvl*100)}", rotation=90, color='red')

plt.title("Wind Generation Reliability Analysis (P-Levels)")
plt.xlabel("Generation (MW)")
plt.legend()
plt.show()

print(f"P95 Reliable Generation: {quantiles[0.05]:.2f} MW")